**Intelligent Vision Systems · Lab 6 · Camera Calibration**
Companion to **Lecture 6 — Camera Calibration**
Dr. Muhammad Kazim · Department of Intelligent Systems, University of Lahore

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kazimbalti/intelligent-vision-systems/blob/main/LAB/Lab_6/01-pinhole-and-distortion.ipynb) &nbsp; [Lecture 6 slides](https://kazimbalti.github.io/intelligent-vision-systems/lectures/06_Camera_Calibration.html#/demo-pinhole) &nbsp; [All Lab 6 notebooks](https://kazimbalti.github.io/intelligent-vision-systems/labs.html#lab-6-calibration)

# Lab 6A · The Pinhole Camera & Lens Distortion

_Lecture 6, Sections 2–3: the pinhole model and lens distortion._

Before we calibrate a real camera we build the **model** ourselves: how a 3-D point becomes a pixel
(`K [R | t]`) and how a lens bends it (`k1, k2, p1, p2, k3`). Everything here is plain NumPy, checked against OpenCV.

In [ ]:
# Setup (only needed on Google Colab or a fresh environment).
# Uncomment the next line to install the packages this notebook uses:
# !pip install -q opencv-python numpy matplotlib

import os, urllib.request

# Fetch the lab files if they are not next to the notebook.
RAW = "https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/LAB/Lab_6"
for f in ['lab6_checks.py']:
    if not os.path.exists(f):
        os.makedirs(os.path.dirname(f) or ".", exist_ok=True)
        urllib.request.urlretrieve(f"{RAW}/{f}", f)
print("data ready")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import lab6_checks as C   # auto-checks for this lab
np.set_printoptions(precision=3, suppress=True)
print("OpenCV", cv2.__version__)

## 1. Project points with the pinhole model

For a camera at the origin looking along +Z (R = I, t = 0):

$$u = f_x \frac{X}{Z} + c_x, \qquad v = f_y \frac{Y}{Z} + c_y$$

**Quiz:** a camera has $f_x = f_y = 800$, $c_x = 640$, $c_y = 360$. Write `project(K, X)` that returns an `(N, 2)` array of
pixels for an `(N, 3)` array of points, using only NumPy.

In [ ]:
K = np.array([[800, 0, 640],
              [0, 800, 360],
              [0,   0,   1]], dtype=float)
X = np.array([[0.5, -0.25, 4.0],    # the slide quiz
              [1.0, -0.50, 8.0],    # twice as far along the same ray
              [-1.0, 1.0, 4.0]])

def project(K, X):
    # TODO: divide by Z, then apply K. Return an (N, 2) array of (u, v).
    uv = np.zeros((len(X), 2))
    return uv

uv = project(K, X)
print(uv)

### ✅ Auto-check
Run the cell below after every change. It prints ✅ when your result is good, or a 💡 hint, never the answer.

In [ ]:
C.check_projection(uv)

<details><summary><b>Show a solution</b> (try first!)</summary>

```python
def project(K, X):
    x = X[:, :2] / X[:, 2:3]                 # normalised coordinates
    return x * [K[0, 0], K[1, 1]] + [K[0, 2], K[1, 2]]
```
</details>

Compare with OpenCV, which does the same thing (plus distortion and a pose):

In [ ]:
uv_cv, _ = cv2.projectPoints(X, np.zeros(3), np.zeros(3), K, None)
print(uv_cv.reshape(-1, 2))

## 2. K from a datasheet

The Raspberry Pi **Camera Module v2** (Sony IMX219): focal length 3.04 mm, pixel size 1.12 µm, 3280 × 2464 pixels.
Compute `fx` (pixels), `cx`, `cy` and the horizontal field of view in degrees.

In [ ]:
f_mm, pixel_um, W, H = 3.04, 1.12, 3280, 2464

# TODO
fx = 0
cx, cy = 0, 0
hfov_deg = 0

print(f"fx = {fx:.1f} px, (cx, cy) = ({cx}, {cy}), HFOV = {hfov_deg:.1f} deg")

In [ ]:
C.check_intrinsics(fx, cx, cy, hfov_deg)

**Think:** you capture at **640 × 480** instead of 3280 × 2464. Which entries of K change, and by how much?
(Also note: 3280 × 2464 is 4:3, and so is 640 × 480, but 1920 × 1080 is a *crop*, so cx, cy change differently.)

## 3. Lens distortion: implement OpenCV's model

With normalised coordinates $(x, y)$ and $r^2 = x^2 + y^2$:

$$x_d = x(1 + k_1 r^2 + k_2 r^4 + k_3 r^6) + 2p_1 xy + p_2(r^2 + 2x^2)$$
$$y_d = y(1 + k_1 r^2 + k_2 r^4 + k_3 r^6) + p_1(r^2 + 2y^2) + 2p_2 xy$$

`k` is in OpenCV's order `[k1, k2, p1, p2, k3]`.

In [ ]:
def distort_points(x, y, k):
    k1, k2, p1, p2, k3 = k
    # TODO: return an (N, 2) array of distorted (x_d, y_d)
    return np.stack([x, y], axis=1)

In [ ]:
C.check_distort(distort_points)

## 4. See the distortion on a grid

Barrel (k1 < 0), pincushion (k1 > 0), mustache (k1 < 0, k2 > 0) and tangential (p1, p2).

In [ ]:
def draw_grid(ax, k, title):
    t = np.linspace(-1, 1, 200)
    for v in np.linspace(-1, 1, 11):
        for xs, ys in ((t, np.full_like(t, v)), (np.full_like(t, v), t)):
            d = distort_points(xs, ys, k)
            ax.plot(d[:, 0], d[:, 1], color="#28568a", lw=1)
    ax.set_title(title); ax.set_aspect("equal"); ax.set_xlim(-1.3, 1.3); ax.set_ylim(1.3, -1.3); ax.axis("off")

cases = [([0, 0, 0, 0, 0], "none"), ([-0.25, 0, 0, 0, 0], "barrel"), ([0.25, 0, 0, 0, 0], "pincushion"),
         ([-0.4, 0.35, 0, 0, 0], "mustache"), ([0, 0, 0.05, 0.04, 0], "tangential")]
fig, ax = plt.subplots(1, 5, figsize=(20, 4.5))
for a, (k, t) in zip(ax, cases): draw_grid(a, k, t)
plt.show()

## 5. A real camera: the GoPro from Lab 6C

Calibration of the Udacity GoPro (Lab 6C) gives roughly `fx = fy = 560`, `(cx, cy) = (651, 499)` at 1280 × 960 and
`dist = [-0.233, 0.062, 0.000, 0.000, -0.008]`. Project a flat, 1 m × 0.75 m grid 0.6 m in front of it:

In [ ]:
Kg = np.array([[560, 0, 651], [0, 561, 499], [0, 0, 1]], float)
Dg = np.array([-0.233, 0.062, 0.0, 0.0, -0.008])
gx, gy = np.meshgrid(np.linspace(-0.5, 0.5, 21), np.linspace(-0.375, 0.375, 16))
P = np.stack([gx.ravel(), gy.ravel(), np.full(gx.size, 0.6)], 1)
ideal, _ = cv2.projectPoints(P, np.zeros(3), np.zeros(3), Kg, None)
real,  _ = cv2.projectPoints(P, np.zeros(3), np.zeros(3), Kg, Dg)
plt.figure(figsize=(9, 6.8))
plt.scatter(*ideal.reshape(-1, 2).T, s=8, c="red", label="pinhole (no lens)")
plt.scatter(*real.reshape(-1, 2).T, s=8, c="#28568a", label="GoPro lens")
plt.gca().add_patch(plt.Rectangle((0, 0), 1280, 960, fill=False, lw=2))
plt.xlim(-300, 1580); plt.ylim(1160, -200); plt.gca().set_aspect("equal"); plt.legend(); plt.title("Same points, with and without the lens")
plt.show()

## Your turn

1. In section 1, move the camera: use `cv2.projectPoints(X, rvec, tvec, K, None)` with `tvec = [0, 0, 2]` and
   `rvec = [0, np.radians(20), 0]`. Explain what happens to each point.
2. In section 5, how many of the red (pinhole) points fall **outside** the image, and how many blue ones? What does that
   tell you about the field of view of a barrel-distorted lens?
3. Change `distort_points` so it only uses k1 (set the others to 0). Plot the GoPro grid again. Where is the difference
   to the full model largest, in pixels?